In [ ]:
import numpy as np


In [ ]:
class KalmanFilter:
    def __init__(self, initial_state, initial_covariance, transition_matrix, measurement_matrix, process_noise_cov, measurement_noise_cov):
        self.state = initial_state
        self.covariance = initial_covariance
        self.F = transition_matrix
        self.H = measurement_matrix
        self.Q = process_noise_cov
        self.R = measurement_noise_cov

    def predict(self):
        self.state = self.F @ self.state
        self.covariance = self.F @ self.covariance @ self.F.T + self.Q

    def update(self, measurement):
        y = measurement - self.H @ self.state
        S = self.H @ self.covariance @ self.H.T + self.R
        K = self.covariance @ self.H.T @ np.linalg.inv(S)
        self.state = self.state + K @ y
        self.covariance = (np.eye(len(self.state)) - K @ self.H) @ self.covariance

    def likelihood(self, measurement):
        y = measurement - self.H @ self.state
        S = self.H @ self.covariance @ self.H.T + self.R
        return np.exp(-0.5 * y.T @ np.linalg.inv(S) @ y) / np.sqrt(2 * np.pi * np.linalg.det(S))


In [ ]:
class SwitchingKalmanFilter:
    def __init__(self, initial_state, initial_covariance, transition_matrix, measurement_matrix, process_noise_covs, measurement_noise_cov, initial_model_probs):
        self.num_models = len(process_noise_covs)  # Number of models (positive and negative trend)
        self.models = []
        for i in range(self.num_models):
            kf = KalmanFilter(initial_state, initial_covariance, transition_matrix, measurement_matrix, process_noise_covs[i], measurement_noise_cov)
            self.models.append(kf)

        self.model_probs = initial_model_probs  # Initial probabilities for each model
        self.transition_probs = np.array([[0.95, 0.05], [0.05, 0.95]]) # Example transition probabilities. Adjust as needed.

    def predict(self, measurement):
        # 1. Prediction Step for Each Model
        for i in range(self.num_models):
            self.models[i].predict()

        # 2. Update Model Probabilities (IMM-like approach)
        likelihoods = np.zeros(self.num_models)
        for i in range(self.num_models):
            likelihoods[i] = self.models[i].likelihood(measurement)

        mixed_model_probs = np.zeros(self.num_models)
        for j in range(self.num_models):
          for i in range(self.num_models):
            mixed_model_probs[j] += self.transition_probs[i][j] * self.model_probs[i]

        for i in range(self.num_models):
            self.model_probs[i] = likelihoods[i] * mixed_model_probs[i]
        self.model_probs /= np.sum(self.model_probs)  # Normalize

        # 3. Combined State Estimate (Weighted average)
        combined_state = np.zeros_like(self.models[0].state)
        for i in range(self.num_models):
            combined_state += self.model_probs[i] * self.models[i].state

        return combined_state, self.model_probs


In [ ]:
# Example Usage:
initial_state = np.array([0.0, 0.0, 0.0])  # [P, r, trend] - trend is new
initial_covariance = np.eye(3) * 0.01

# Transition matrix now includes trend:
transition_matrix = np.array([[1.0, 1.0, 0.0],  # P += r
                              [0.0, 1.0, 1.0],  # r += trend
                              [0.0, 0.0, 1.0]])  # trend evolves (could be 1.0 if constant)

measurement_matrix = np.array([[1.0, 0.0, 0.0]])  # Observing P

# Process noise for positive and negative trend models:
process_noise_covs = [np.diag([0.0001, 0.00001, 0.000001]),  # Positive trend (smaller noise for r)
                     np.diag([0.0001, 0.00001, 0.000001])]  # Negative trend (smaller noise for r)

measurement_noise_cov = np.array([[0.01]])  # Measurement noise

initial_model_probs = np.array([0.5, 0.5])  # Initially equal probability for both models

skf = SwitchingKalmanFilter(initial_state, initial_covariance, transition_matrix, measurement_matrix, process_noise_covs, measurement_noise_cov, initial_model_probs)

# Example measurements (replace with your actual data)
measurements = [0.1, 0.2, 0.3, 0.4, 0.5, 0.4, 0.3, 0.2, 0.1, 0.0, -0.1, -0.2, -0.3, -0.4, -0.5, -0.4, -0.3, -0.2, -0.1, 0.0, 0.1, 0.2, 0.3]  # Example with trend change

for measurement in measurements:
    estimated_state, model_probs = skf.predict(np.array([measurement]))
    print("Estimated state:", estimated_state, "Model Probabilities:", model_probs)

```python
import numpy as np

class SwitchingKalmanFilter:
    def __init__(self, initial_state, initial_covariance, transition_matrix, measurement_matrix, process_noise_covs, measurement_noise_cov, initial_model_probs):
        self.num_models = len(process_noise_covs)  # Number of models (positive and negative trend)
        self.models = []
        for i in range(self.num_models):
            kf = KalmanFilter(initial_state, initial_covariance, transition_matrix, measurement_matrix, process_noise_covs[i], measurement_noise_cov)
            self.models.append(kf)

        self.model_probs = initial_model_probs  # Initial probabilities for each model
        self.transition_probs = np.array([[0.95, 0.05], [0.05, 0.95]]) # Example transition probabilities. Adjust as needed.

    def predict(self, measurement):
        # 1. Prediction Step for Each Model
        for i in range(self.num_models):
            self.models[i].predict()

        # 2. Update Model Probabilities (IMM-like approach)
        likelihoods = np.zeros(self.num_models)
        for i in range(self.num_models):
            likelihoods[i] = self.models[i].likelihood(measurement)

        mixed_model_probs = np.zeros(self.num_models)
        for j in range(self.num_models):
          for i in range(self.num_models):
            mixed_model_probs[j] += self.transition_probs[i][j] * self.model_probs[i]

        for i in range(self.num_models):
            self.model_probs[i] = likelihoods[i] * mixed_model_probs[i]
        self.model_probs /= np.sum(self.model_probs)  # Normalize

        # 3. Combined State Estimate (Weighted average)
        combined_state = np.zeros_like(self.models[0].state)
        for i in range(self.num_models):
            combined_state += self.model_probs[i] * self.models[i].state

        return combined_state, self.model_probs

class KalmanFilter:  # Standard Kalman filter (same as before)
    # ... (Implementation remains the same)

# Example Usage:
initial_state = np.array([0.0, 0.0, 0.0])  # [P, r, trend] - trend is new
initial_covariance = np.eye(3) * 0.01

# Transition matrix now includes trend:
transition_matrix = np.array([[1.0, 1.0, 0.0],  # P += r
                              [0.0, 1.0, 1.0],  # r += trend
                              [0.0, 0.0, 1.0]])  # trend evolves (could be 1.0 if constant)

measurement_matrix = np.array([[1.0, 0.0, 0.0]])  # Observing P

# Process noise for positive and negative trend models:
process_noise_covs = [np.diag([0.0001, 0.00001, 0.000001]),  # Positive trend (smaller noise for r)
                     np.diag([0.0001, 0.00001, 0.000001])]  # Negative trend (smaller noise for r)

measurement_noise_cov = np.array([[0.01]])  # Measurement noise

initial_model_probs = np.array([0.5, 0.5])  # Initially equal probability for both models

skf = SwitchingKalmanFilter(initial_state, initial_covariance, transition_matrix, measurement_matrix, process_noise_covs, measurement_noise_cov, initial_model_probs)

# Example measurements (replace with your actual data)
measurements = [0.1, 0.2, 0.3, 0.4, 0.5, 0.4, 0.3, 0.2, 0.1, 0.0, -0.1, -0.2, -0.3, -0.4, -0.5, -0.4, -0.3, -0.2, -0.1, 0.0, 0.1, 0.2, 0.3]  # Example with trend change

for measurement in measurements:
    estimated_state, model_probs = skf.predict(np.array([measurement]))
    print("Estimated state:", estimated_state, "Model Probabilities:", model_probs)

```

**Key Changes and Explanations:**

1. **State Vector:** The state vector is now `[P, r, trend]`.  `trend` represents the overall direction of change in `r`.

2. **Transition Matrix:** The transition matrix is updated to include the `trend` component:
   ```
   F = [[1.0, 1.0, 0.0],  // P += r
        [0.0, 1.0, 1.0],  // r += trend
        [0.0, 0.0, 1.0]]  // trend evolves (could be 1.0 if constant)
   ```

3. **Process Noise:**  The `process_noise_covs` is now a *list* of covariance matrices, one for each model (positive and negative trend). You would adjust the noise values, especially for `r`, to reflect the different behavior in each regime.  You might have smaller noise on `r` when it is trending in one direction, and larger noise when flipping direction.

4. **`SwitchingKalmanFilter` Class:**  This class now manages the two Kalman filters (one for each trend) and performs the switching logic using an IMM-like approach.

5. **Model Probabilities:**  The `model_probs` attribute stores the probabilities of each model.

6. **Transition Probabilities:** The `transition_probs` matrix defines the probabilities of switching between models. You'll need to adjust these based on how often you expect direction flips.

7. **Combined Estimate:** The final state estimate is a weighted average of the estimates from both models, weighted by their probabilities.

8. **Example Usage:**  The example demonstrates how to use the `SwitchingKalmanFilter` with example measurements that include a direction flip.

**How to Adapt:**

* **Transition Probabilities:**  The most crucial part is to correctly set the `transition_probs`.  These probabilities determine how often the filter switches between the models.  If direction flips are frequent, increase the off-diagonal elements (e.g., `transition_probs[0][1]` and `transition_probs[1][0]`).  If they are rare, decrease them.
* **Process Noise:** Adjust the `process_noise_covs` to reflect the uncertainty in each model.  The model with a more stable trend for `r` should have a smaller process noise for `r`.
* **Trend Model:**  The current implementation assumes the `trend` can change slowly.  You could modify the transition matrix for `trend` if you have a more specific model for how the trend evolves over time.

This improved code provides a more robust and accurate way to handle direction flips in your process using a switching Kalman filter.  Remember to carefully tune the parameters to your specific needs.
